In [39]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import sys
import os
import time
import random
import csv
import re
import json

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}
COLS = ['id', 'price', 'city', 'lat', 'lon', 'bathrooms', 'height',
        'state', 'type', 'year', 'floor', 'max_floor', 'area', 'views']

In [45]:
def prepare_entry(r2, id_):
    result = []
    if r2 is None or r2.status_code != 200:
        return None
    html = r2.text
    soup2 = BeautifulSoup(html, "lxml")

    result.append(str(id_))

    price_el = soup2.select_one("div.offer__price")
    if price_el is None:
        cur_title = soup2.select_one("title")
        if cur_title:
            m = re.search(r"\s[0-9]+\s", cur_title.get_text())
            result.append(m.group(0).strip() if m else None)
        else:
            return None
    else:
        price = price_el.get_text(strip=True).replace("\xa0", "").replace("〒", "")
        result.append(price)

    location_el = soup2.select_one("div.offer__location")
    if location_el is None:
        return None
    city_text = location_el.get_text(strip=True)
    city_match = re.search(r"^[^,]+", city_text)
    result.append(city_match.group() if city_match else city_text)

    lat = re.search(r'"lat"\s*:\s*([0-9.]+)', html)
    lon = re.search(r'"lon"\s*:\s*([0-9.]+)', html)
    if not lat or not lon:
        return None
    result += [lat.group(1), lon.group(1)]

    bathrooms = re.search(r"Санузел(\d)", html)
    if bathrooms:
        result.append(bathrooms.group(1))
    else:
        alt = re.search(r"(\d)\s*с/у", html)
        result.append(alt.group(1) if alt else "1")

    height_m = re.search(r"(\d+\.\d+)\s*м</dd>", html)
    result.append(height_m.group(1) if height_m else "2")

    items = soup2.select("div.offer__info-item")
    text = re.sub(r"\s+", " ", " ".join(i.get_text(" ", strip=True) for i in items).replace("\xa0", " ")).strip()
    state_m = re.search(r"квартиры\s*(.*)", text, re.I)
    result.append(state_m.group(1).strip() if state_m else "черновая отделка")

    type_val, year_val, area_val, floor_val, maxfloor_val = None, None, None, None, None
    for cnt, i in enumerate(soup2.select("div.offer__advert-short-info"), 1):
        t = i.get_text(strip=True)
        if cnt == 2:
            type_val = t
        if m := re.search(r"\b(19|20)\d{2}\b", t):
            year_val = m.group()
        if a := re.search(r"(\d+)\s*м²", t):
            area_val = a.group(1)
        if f := re.search(r"(\d+)\s*из\s*(\d+)", t):
            floor_val, maxfloor_val = f.group(1), f.group(2)

    result += [type_val, year_val, floor_val, maxfloor_val, area_val]

    try:
        r = requests.get(f"https://krisha.kz/ms/views/krisha/live/{id_}/",
                         params={"id": id_}, headers=HEADERS, timeout=10)
        data = r.json()
        nb_p = data["data"][str(id_)]["nb_phone_views"]
        nb_v = data["data"][str(id_)]["nb_views"]
        result.append(nb_p + nb_v)
    except Exception:
        result.append(0)

    if len(result) < len(COLS):
        result += [None] * (len(COLS) - len(result))
    return result[:len(COLS)]

In [46]:
r = requests.get(f"https://krisha.kz/")
bs = BeautifulSoup(r.text, "lxml")

In [47]:
all_ids = [a.get('data-id') for a in bs.find_all('a') if a.has_attr('data-id')]
all_ids = [i for i in all_ids if len(i) == 10]
print(all_ids)

['1015441531', '1011564806', '1000673584', '1010521161', '1006792996', '1014301916', '1012282091', '1014205767', '1015616360', '1003825969', '1013039623', '1009004608', '1015478957', '1001409549', '1003452509', '1005972194', '1015825553', '1003113336', '1014981885', '1016083712', '1015183044']


In [ ]:
with open('./dirty_data.csv', mode='a', encoding='utf-8', newline='') as file:
    writer = csv.writer(file)
    for id in all_ids[:7]:
        link = f"https://krisha.kz/a/show/{id}"
        r = requests.get(link)
        res = prepare_entry(r, id)
        
        if res is not None:
            print("OK", res)
            writer.writerow(res)
        else:
            print("SKIP")
        time_sleep = np.random.random()
        #time.sleep(time_sleep)


OK ['1015441531', '23000000₸', 'Каратобе', '43.429732460393', '76.739697378712', '1', '2', 'черновая отделка', 'Да', '2020', None, None, '20', 158]
OK ['1011564806', '23938860', 'Астана', '51.12417530114', '71.386014740376', '1', '2', 'черновая отделка', 'монолитный', '2025', None, None, '7', 6725]
SKIP
OK ['1010521161', '41553090', 'Астана', '51.081590143885', '71.494511203977', '1', '2', 'черновая отделка', 'монолитный', '2026', None, None, '53', 3291]
OK ['1006792996', '19000000₸', 'Аскара Токпанова (КазЦИК)', '43.412717829843', '76.848100225432', '1', '2.8', 'черновая отделка', 'Нет', '2024', None, None, '20', 4067]
OK ['1014301916', '99800000₸', 'Алматы', '43.316151343586', '76.985587634828', '1', '2', 'черновая отделка', 'Нет', '2012', None, None, '5', 2920]
SKIP
